# Practical 4: Hand-eye coordination

The goal of this practical session is learning how to grasp objects based on 2D points detected in camera images.

In this practical you will do the following:
* Run our hand-eye calibration tool to find the transformation between the camera and the robot TCP.
* Do triangulation to find the 3D position of a point in the camera frame, and transform it to the robot's base frame.
* Define an object frame for a bin object, and determine how to find the bin's pose from detected keypoints
* Define a grasp frame relative to the bin's frame
* Execute your grasp and see whether it works


<div class="alert alert-block alert-success">
Note, take pictures of the microplate in different settings on your manipulation table for the next homework and practical! You will have to train a keypoint detector with this data. (not in this practical, but the next homework!). 
</div>

## 1. Introduction

### 1.1 Theory review
Pinholes model, triangulation, hand-eye calibration were all explained during the lectures. Make sure you have familiarized yourself with these concepts and ask questions if you haven't yet understood everything. 



<div class="alert alert-block alert-info"> <b>Task 1:</b> Run the cells below to install and import the dependencies.</div>


In [ ]:
!current_env=$(conda info --envs | grep "*" | awk '{print $1}'); if [ "$current_env" = "irm" ]; then echo "The current active Conda environment is 'irm'. You can run the next cell."; else echo "The current active Conda environment is not 'irm'. Make sure to run this notebook with your irm conda env (conda activate irm)"; fi

In [ ]:
!pip install numpy~=1.26 # open3D requires numpy < 2.0 for now
!pip install open3D

In [ ]:
from typing import Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
from airo_camera_toolkit.utils.image_converter import ImageConverter
from airo_camera_toolkit.cameras.realsense.realsense import Realsense
from airo_dataset_tools.data_parsers.pose import Pose
from airo_robots.grippers.hardware.robotiq_2f85_urcap import Robotiq2F85
from airo_robots.manipulators.hardware.ur_rtde import URrtde
from airo_spatial_algebra import SE3Container
from airo_typing import (
    CameraIntrinsicsMatrixType,
    HomogeneousMatrixType,
    NumpyDepthMapType,
    NumpyIntImageType,
    Vector2DType,
    Vector3DType,
    JointConfigurationType,
)
from airo_robots.manipulators.position_manipulator import PositionManipulator
from airo_camera_toolkit.interfaces import RGBDCamera

## 2. Hand-eye calibration

![Wrist camera](https://i.imgur.com/dvuFB60.png)

<div class="alert alert-block alert-info"> <b>Task 2:</b> Try to guess the pose of the camera in the TCP frame(which is now defied as the tip of the robot gripper fingers, as illustrated in the image above). The RGB module of the realsense camera is the smallest eye. 
</div>


In [ ]:
X_TCP_C = np.identity(4) 

# TODO @students: edit X_TCP_C to represent the pose of the camera in the TCP frame


with np.printoptions(precision=3, suppress=True):
    print(X_TCP_C)

We will now determine this pose more accurately using *hand-eye calibration*. We have provided a command line interface for doing this, which you will now explore.
<div class="alert alert-block alert-info"> <b>Task 3:</b> Run the cell below to see how to run the hand-eye calibration. </div>


 <!-- Save the result in this directory as <b>camera_pose.json</b> Run the cell below and compare with your guess from the previous task, were you close? </div>


Read the <a href="https://github.com/airo-ugent/airo-mono/blob/main/airo-camera-toolkit/airo_camera_toolkit/calibration/README.md">airo-camera-toolkit calibration documentation</a> to find out which type of calibration you need <b>eye_in_hand</b> or <b>eye_to_hand</b>. </div>

<div class="alert alert-block alert-info"> <b>Task 3:</b> Open a terminal and activate your conda environment.  </div>
 how to run the hand-eye calibration. Save the result in this directory as <b>camera_pose.json</b> Run the cell below and compare with your guess from the previous task, were you close? </div> -->

In [ ]:
!airo-camera-toolkit hand-eye-calibration --help

<div class="alert alert-block alert-info"> <b>Task 4:</b> Read the <a href="https://github.com/airo-ugent/airo-mono/blob/main/airo-camera-toolkit/airo_camera_toolkit/calibration/README.md">documentation</a> to find out which type of calibration you need: <b>eye_in_hand</b> or <b>eye_to_hand</b>. Complete the command with the proper mode and the robot's IP address. Then run the calibration.  </div>

Take about 5 images and make sure to vary the orientation of the camera with respect to the frame as much as possible.

Navigate to the folder called `calibration_<datetime>/results_n=5`. There is one result (json file) per available solver, choose the best result (lowest residual error) and save it in the parent directory of this notebook as <b>camera_pose.json</b>.

 You can ignore possible warnings like QObject:moveToThread. Scroll down passed these warnings to see the intermediate output of the calibration process.

In [ ]:
#!airo-camera-toolkit hand-eye-calibration --robot_ip 10.42.0.162 --mode eye_in_hand 


<div class="alert alert-block alert-info"> <b>Task 5:</b> Run the cell below and compare with your guess for the extrinsics.

In [ ]:
camera_pose_path = "camera_pose.json"

with open(camera_pose_path, "r") as f:
    camera_pose = Pose.model_validate_json(f.read()).as_homogeneous_matrix()

with np.printoptions(precision=3, suppress=True):
    print(camera_pose)
    print(f"Distance between guessed camera position and camera position from the calibration procedure: {np.linalg.norm(X_TCP_C[:3,3]-camera_pose[:3,3]):0.03f} m")
    

Is your guessed camera position close to the camera position as calculated by the calibration procedure? How accurate do you think the calibration procedure is (< 1 mm, < 1 cm, < 5 cm, < 10 cm)? What does the extrinsics matrix tell you about the orientation of the camera? 

<div class="alert alert-block alert-info"> <b>Task 6:</b> Answer the questions below: </div>

@student answer

## 3. Triangulation from multiple views

### 3.1 Creating the camera, robot and grippper objects



### Gripper Initialization

In this practical, we will use the RObotiq 2F-85 for the first time. 

The grippers are connected to the Robot wrist using an RS-485 physical interface. The gripper driver communicates over the ModbusRTU protocol, but we have abstracted the communication with the gripper in our *airo-mono* stack. You can simply control it by creating a new gripper object and sending the 'open' and 'close' commands:


```python
gripper = Robotiq2F85(robot_ip_address)
gripper.open()
gripper.close()
```


There is one thing you need to check first though. 
We defined the TCP frame as the frame of the tip of the tool that is attached to the robot, so we need to let the robot controlbox know about the details of our gripper. Specifically we need to specify the position of the tcp frame origin in the robot flange frame. To facilitate freedriving, the robot also needs to know how heavy the gripper is so that it can counteract the gravity, therefore we also need to specify the weight and center of mass. 



<div class="alert alert-block alert-info"> <b>Task 7:</b> 
Take a look in the datasheet of the 2F-85: https://assets.robotiq.com/website-assets/support_documents/document/2F-85_2F-140_Instruction_Manual_e-Series_PDF_20190206.pdf#page=124 and verify if the CoM, mass and offset are configured corectly in the robot controller. 


</div>

You can find the offset under `Installation > TCP` and the payload details under ` Installation > Payload ` on the robot teach pendant.




<div class="alert alert-block alert-info"> <b>Task 8:</b> Run the cells below to create the camera, robot and gripper. </div>

In [ ]:
try:
    camera = Realsense()
except RuntimeError as e:
    if "busy" in str(e):
        print("Camera is busy. probably you have already created the camera object. you can ignore this error")
    else:
        raise e

with np.printoptions(precision=1, suppress=True):
    print("Camera intrinsics: \n", camera.intrinsics_matrix())
    print("Camera resolution:", camera.resolution)

In [ ]:
image = camera.get_rgb_image_as_int()

with np.printoptions(precision=1, suppress=True):
    print("Image shape:", image.shape)
    print("Image dtype:", image.dtype)


plt.imshow(camera.get_rgb_image_as_int());

In [ ]:
ip_address = None  # fill in the robot IP
robot = URrtde(ip_address, URrtde.UR3E_CONFIG)
gripper = Robotiq2F85(ip_address)
gripper.close().wait()
gripper.open().wait()

with np.printoptions(precision=1, suppress=True):
    print("Robot joint configuration:", np.rad2deg(robot.get_joint_configuration()), "degrees")
    print(f"Gripper width: {gripper.get_current_width():.4f} m")

### 3.2 Collecting `(tcp_pose, image)` pairs

<div class="alert alert-block alert-info"> <b>Task 9:</b> Place the red bin on your robot station, on a place where the robot can realistically grasp it. Move the robot in Freedrive to choose two viewpoints for looking at the bin. Try to have the projection lines perpendicular. Fill in the joint angles for these views below:</div>

To verify the bin is within the camera view,  visualize the current view of the camera in the cell below and run it while freedriving the robot before writing down the joint configuration.

Don't forget to put the robot back in remote control once your are finished if you used freedrive on the teaching pendant instead of the commands in the notebook. 


In [ ]:
robot.rtde_control.teachMode()

In [ ]:
current_img = None #TODO @students: get the current image from the camera
plt.imshow(current_img);
print(f"joint config for this picture: {robot.get_joint_configuration()}\n(you can copy paste this two cells under this one)")

In [ ]:
robot.rtde_control.EndTeachMode()

In [ ]:
# TODO: @students: set fill in joint angles for the two views
joints_view_A = None # np.array([0,0,0,0,0,0])
joints_view_B =  None # np.array([0,0,0,0,0,0])

<div class="alert alert-block alert-info"> <b>Task 10:</b> Keep the emergency stop close and run the cell below to move the robot to the viewpoints. </div>

In [ ]:
def collect_rgbd_and_tcp_pose(
    robot: PositionManipulator, camera: RGBDCamera, joints: JointConfigurationType
) -> Tuple[NumpyIntImageType, NumpyDepthMapType, HomogeneousMatrixType]:
    robot.move_to_joint_configuration(joints).wait()
    image = camera.get_rgb_image_as_int()
    depth_map = camera._retrieve_depth_map()
    X_B_TCP = robot.get_tcp_pose()
    return image, depth_map, X_B_TCP


image_A, depth_map_A, X_B_TCPA = collect_rgbd_and_tcp_pose(robot, camera, joints_view_A)
image_B, depth_map_B, X_B_TCPB = collect_rgbd_and_tcp_pose(robot, camera, joints_view_B)


def imshow_left_right(image_left, image_right):
    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.imshow(image_left)
    plt.title("Left image")
    plt.subplot(1, 2, 2)
    plt.imshow(image_right)
    plt.title("Right image")
    plt.show()


imshow_left_right(image_A, image_B)

<div class="alert alert-block alert-info"> <b>Task 11:</b> Run the cell and click a point on the box (the same in the left and right image). We will move the gripper there later in this practical so take a point that is touchable by the robot gripper. The point should ideally be clearly visible in both images. </div>

In [ ]:
cyan = (255, 255, 0)  # default color for left image
magenta = (255, 0, 255)  # default color for right image


def draw_point_coordinate(image, point, color):
    """Draws a circle on the image at the given point with the coordinates next to it."""
    text = f"({point[0]}, {point[1]})"
    text_location = (point[0] + 12, point[1] - 12)
    cv2.circle(image, point, 1, color, 1)
    cv2.circle(image, point, 15, color, 4)
    cv2.putText(image, text, text_location, cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3, cv2.LINE_AA)


def annotate_multiview(
    left_image: NumpyIntImageType,
    right_image: NumpyIntImageType,
    color_left: Tuple[int, int, int] = cyan,
    color_right: Tuple[int, int, int] = magenta,
):
    left_point = [None]
    right_point = [None]

    def on_click_left(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            print(f"left click: (u,v) =  ({x}, {y})")
            left_point[0] = (x, y)

    def on_click_right(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            print(f"right click: (u,v) =  ({x}, {y})")
            right_point[0] = (x, y)

    cv2.namedWindow("Left Image", cv2.WINDOW_NORMAL)
    cv2.namedWindow("Right Image", cv2.WINDOW_NORMAL)
    cv2.setMouseCallback("Left Image", on_click_left)
    cv2.setMouseCallback("Right Image", on_click_right)

    # resize the windows and place them side by side
    cv2.resizeWindow("Left Image", 800, 600)
    cv2.resizeWindow("Right Image", 800, 600)
    cv2.moveWindow("Left Image", 0, 0)
    cv2.moveWindow("Right Image", 800, 0)

    left_image_bgr = ImageConverter.from_numpy_int_format(left_image).image_in_opencv_format
    right_image_bgr = ImageConverter.from_numpy_int_format(right_image).image_in_opencv_format

    print("Press 'Enter' or 'q' to finish.")
    while True:
        left_annotated = left_image_bgr.copy()
        right_annotated = right_image_bgr.copy()

        if left_point[0] is not None:
            draw_point_coordinate(left_annotated, left_point[0], color_left)

        if right_point[0] is not None:
            draw_point_coordinate(right_annotated, right_point[0], color_right)

        cv2.imshow("Left Image", left_annotated)
        cv2.imshow("Right Image", right_annotated)
        key = cv2.waitKey(1)

        if key == ord("q") or key == 13:
            break

    print("Left point: ", left_point[0])
    print("Right point: ", right_point[0])
    cv2.destroyAllWindows()
    return left_point[0], right_point[0]


point_A, point_B = annotate_multiview(image_A, image_B)

### 3.2 Triangulating the 3D point

<div class="alert alert-block alert-info"> <b>Task 12:</b> To run the triangulation, we need to provide the pose of the camera when image_A and image_B were taken <b>expressed in the robot base frame</b>. Calculate these poses in in the cell below. </div>

In [ ]:
X_TCP_C = camera_pose

# TODO @students: calculate the camera pose in the base frame for view A and view B
X_B_CA = None # CA stands for "camera pose when image A was taken"
X_B_CB = None

In the cell below we give you our solution to the `least_squares_triangulation()` function from the homework, but feel free to test your own version!

In [ ]:
def least_squares_triangulation(
    point_left: Vector2DType,
    point_right: Vector2DType,
    intrinsics: CameraIntrinsicsMatrixType,
    left_extrinsics: HomogeneousMatrixType,
    right_extrinsics: HomogeneousMatrixType,
) -> np.ndarray:
    """Triangulates a 3D point from a pair of 2D points in the left and right image

    Args:
        point_left: (u,v) pixel coordinate of the point in the left image
        point_right: (u,v) pixel coordinates of the point in the right image
        intrinsics: the camera's intrinsics matrix
        left_extrinsics: camera pose for the left image
        right_extrinsics: camera pose for the right image (must be expressed in the same frame as the left pose])

    Returns:
        the 3D point (expressed in the same frame as the camera poses)
    """

    def _to_homogeneous(np_array):
        return np.append(np_array, 1)

    # create the rays in the camera frome from the pixel coordinates
    ray_left = np.linalg.inv(intrinsics) @ _to_homogeneous(np.array(point_left))
    ray_right = np.linalg.inv(intrinsics) @ _to_homogeneous(np.array(point_right))

    # transform the rays to the world frame
    ray_left_in_world = left_extrinsics[:3, :3] @ ray_left
    ray_right_in_world = right_extrinsics[:3, :3] @ ray_right

    # reshape the rays to be column vectors
    ray_right_in_world = ray_right_in_world.reshape(3, 1)
    ray_left_in_world = ray_left_in_world.reshape(3, 1)

    # create the least squares LHS and RHS
    lhs = 0
    rhs = 0
    for ray, camera_position in zip(
        (ray_left_in_world, ray_right_in_world), (left_extrinsics[:3, 3], right_extrinsics[:3, 3])
    ):
        lhs += np.eye(3) - ray @ ray.T / np.linalg.norm(ray) ** 2
        rhs += (np.eye(3) - ray @ ray.T / np.linalg.norm(ray) ** 2) @ camera_position

    return np.linalg.inv(lhs) @ rhs

<div class="alert alert-block alert-info"> <b>Task 13:</b> Run the cell below to triangulate the 3D point, and verify in the visualization if it corresponds to the 2D points you clicked. </div>

In [ ]:
from typing import Optional


def open3d_point(
    position: Vector3DType, color: Tuple[float, float, float], radius: float = 0.005
) -> o3d.geometry.TriangleMesh:
    """Creates a small sphere mesh for visualization in open3d.

    Args:
        position:  position of the point
        color: RGB color of the point
        radius: radius of the sphere

    Returns:
        sphere: a mesh
    """
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=radius)
    sphere.translate(position)
    sphere.paint_uniform_color(color)
    sphere.compute_vertex_normals()
    return sphere


def open3d_camera(
    camera_pose: HomogeneousMatrixType,
    intrinsics: CameraIntrinsicsMatrixType,
    resolution: Tuple[int, int],
    color: Optional[Tuple[float, float, float]] is None,
) -> o3d.geometry.LineSet:
    """Creates a camera visualization in open3d.

    Args:
        camera_pose: camera pose
        intrinsics: camera intrinsics
        resolution: width, height of the camera's images

    Returns:
        camera_lines: a line set
    """

    camera_lines = o3d.geometry.LineSet.create_camera_visualization(
        *resolution, intrinsics, np.linalg.inv(camera_pose), scale=0.1
    )
    if color is not None:
        camera_lines.paint_uniform_color(color)

    return camera_lines


def open3d_ray(direction, origin=np.array([0, 0, 0])):
    c = 0.001
    ray_mesh = o3d.geometry.TriangleMesh.create_arrow(c, 4 * c, 1.5, 5 * c)

    rotation_Z = direction
    rotation_X = np.cross(rotation_Z, np.array([0, 1, 0]))
    rotation_Y = np.cross(rotation_Z, rotation_X)
    rotation_matrix = np.column_stack([rotation_X, rotation_Y, rotation_Z])

    ray_mesh.rotate(rotation_matrix, center=(0, 0, 0))

    ray_mesh.translate(origin)

    return ray_mesh


def open3d_ray_from_pixel(
    pixel: Vector2DType,
    camera_pose: HomogeneousMatrixType,
    intrinsics: CameraIntrinsicsMatrixType,
    color: Optional[Tuple[float, float, float]] = None,
) -> o3d.geometry.TriangleMesh:
    """Creates a ray visualization in open3d.

    Args:
        pixel: pixel coordinate of the ray
        camera_pose: camera pose
        intrinsics: camera intrinsics

    Returns:
        ray_mesh: a triangle mesh
    """

    point_homogeneous = np.array([pixel[0], pixel[1], 1])
    ray_direction = np.linalg.inv(intrinsics) @ point_homogeneous
    ray_direction_in_world = camera_pose[:3, :3] @ ray_direction
    ray_origin_in_world = camera_pose[:3, 3]

    ray_mesh = open3d_ray(ray_direction_in_world, ray_origin_in_world)

    if color is not None:
        ray_mesh.paint_uniform_color(color)

    return ray_mesh


def open3d_pointcloud_from_image_and_depth_map(
    image: NumpyIntImageType,
    depth_map: NumpyDepthMapType,
    intrinsics: CameraIntrinsicsMatrixType,
    camera_pose: HomogeneousMatrixType,
    resolution: Tuple[int, int],
) -> o3d.geometry.PointCloud:
    image_o3d = o3d.geometry.Image(image.copy())
    depth_map_o3d = o3d.geometry.Image(depth_map)
    rgbd_o3d = o3d.geometry.RGBDImage.create_from_color_and_depth(
        image_o3d, depth_map_o3d, depth_scale=1.0, depth_trunc=100.0, convert_rgb_to_intensity=False
    )
    intrinsics_o3d = o3d.camera.PinholeCameraIntrinsic(resolution[0], resolution[1], intrinsics)

    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_o3d, intrinsics_o3d, extrinsic=np.linalg.inv(camera_pose)
    )
    return pcd


## open3D helper functions
def visualize_open3D(geometry_list):
    o3d.visualization.draw_geometries(
        geometry_list,
        zoom=0.2,
        front=[1.0, 0.0, 1.0],
        lookat=[0.0, -0.3, 0.0],
        up=[0.0, 0, 1.0],
    )


yellow = (1.0, 1.0, 0.0)
cyan = (0.0, 1.0, 1.0)
magenta = (1.0, 0.0, 1.0)


point_3D = least_squares_triangulation(point_A, point_B, camera.intrinsics_matrix(), X_B_CA, X_B_CB)


camera_A_lines = open3d_camera(X_B_CA, camera.intrinsics_matrix(), camera.resolution, cyan)
camera_B_lines = open3d_camera(X_B_CB, camera.intrinsics_matrix(), camera.resolution, magenta)

# We use image_B and depth_map_B to create the pointcloud, you can change this to A if you want:
pcd = open3d_pointcloud_from_image_and_depth_map(
    image_B, depth_map_B, camera.intrinsics_matrix(), X_B_CB, camera.resolution
)

ray_A_mesh = open3d_ray_from_pixel(point_A, X_B_CA, camera.intrinsics_matrix(), cyan)
ray_B_mesh = open3d_ray_from_pixel(point_B, X_B_CB, camera.intrinsics_matrix(), magenta)


point_3D_mesh = open3d_point(point_3D, yellow)
visualize_open3D([pcd, point_3D_mesh, camera_A_lines, ray_A_mesh, camera_B_lines, ray_B_mesh])

point_3D

<div class="alert alert-block alert-info"> <b>Task 14:</b> Define a pose where the robot (almost) touches the 3D point from above. Tip: use the coordinate system markers on your desk. </div>

In [ ]:
# TODO @students: create the touch pose
p_B_TCP_touch = None   # the position where the TCP will (barely) touch your 3D point (note, we  add np.array([0.0, 0.0, 0.02]) to avoid colliding)
R_B_TCP_touch_X = None # rotation of TCP around X-axis  
R_B_TCP_touch_Y = None  # rotation of TCP around Y-axis  
R_B_TCP_touch_Z = None # rotation of TCP around Z-axis  


R_B_TCP_touch = np.column_stack([R_B_TCP_touch_X, R_B_TCP_touch_Y, R_B_TCP_touch_Z])
assert np.isclose(np.linalg.inv(R_B_TCP_touch) @ R_B_TCP_touch, np.eye(3)).all() and np.linalg.det(R_B_TCP_touch) == 1, "R_B_TCP_touch is not a valid rotation matrix"



<div class="alert alert-block alert-info"> <b>Task 15:</b> Explain why the assert statement is suitable to test if the rotation matrix is valid..</div>
#TODO: @students, answer here.

In [ ]:
X_B_TCP_touch_se3 = SE3Container.from_orthogonal_base_vectors_and_translation(
    R_B_TCP_touch_X, R_B_TCP_touch_Y, R_B_TCP_touch_Z, p_B_TCP_touch
)
X_B_TCP_touch = X_B_TCP_touch_se3.homogeneous_matrix

X_B_TCP_touch[:3, 3] = X_B_TCP_touch[:3, 3] + np.array([0.0, 0.0, 0.01]) # add 1cm to the z-axis to avoid colliding

touch_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
touch_mesh.transform(X_B_TCP_touch)

# TODO visualize
visualize_open3D([pcd, touch_mesh])

<div class="alert alert-block alert-info"> <b>Task 16:</b> Execute the touch by running the cell below.</div>

In [ ]:
gripper.close().wait()
robot.move_to_tcp_pose(X_B_TCP_touch).wait()

If all went as expected, the robot's TCP is now almost touching your 3D point. Make sure that it is fairly accurate, the tcp should be no more than 5mm away from the object in the X and Y directions. If not, you have labeled the corresponding points imprecisely or your camera calibration is not precise enough.

## 4. Object frame from keypoints
![bin](https://i.imgur.com/CMQlsPH.jpg)

When the robot interacts with known rigid objects, it is common to define an **object frame** (or body frame), which is a local coordinate system attached to the object. This allows you to define other points and frame relative to the object.

<div class="alert alert-block alert-info"> <b>Task 17:</b> Make a drawing of the bin (on paper) and choose 3 keypoints that can be use to calculate the object frame shown above. Color your keypoints red, green and blue, we will use this for the visualization. </div>

<!-- [airo-mono reference for coordinate systems](https://github.com/airo-ugent/airo-mono/blob/main/airo-dataset-tools/docs/pose.md#coordinate-system-conventions-world_map) -->

### 4.1 Collecting new images

<div class="alert alert-block alert-info"> <b>Task 18:</b> Run the 3 cells below to collect new images and annotate the three keypoints and visualize the 3D triangulated points.</div>

In [ ]:
image_A, depth_map_A, X_B_TCPA = collect_rgbd_and_tcp_pose(robot, camera, joints_view_A)
image_B, depth_map_B, X_B_TCPB = collect_rgbd_and_tcp_pose(robot, camera, joints_view_B)

imshow_left_right(image_A, image_B)

In [ ]:
# Recalculate the camera poses in case the joint_angles have changed
X_B_CA = X_B_TCPA @ X_TCP_C
X_B_CB = X_B_TCPB @ X_TCP_C

red_cv2 = (0, 0, 255)
green_cv2 = (0, 255, 0)
blue_cv2 = (255, 0, 0)

keypoint_red_A, keypoint_red_B = annotate_multiview(image_A, image_B, color_left=red_cv2, color_right=red_cv2)
keypoint_green_A, keypoint_green_B = annotate_multiview(image_A, image_B, color_left=green_cv2, color_right=green_cv2)
keypoint_blue_A, keypoint_blue_B = annotate_multiview(image_A, image_B, color_left=blue_cv2, color_right=blue_cv2)

intrinsics = camera.intrinsics_matrix()
keypoint_red_3d = least_squares_triangulation(keypoint_red_A, keypoint_red_B, intrinsics, X_B_CA, X_B_CB)
keypoint_green_3d = least_squares_triangulation(keypoint_green_A, keypoint_green_B, intrinsics, X_B_CA, X_B_CB)
keypoint_blue_3d = least_squares_triangulation(keypoint_blue_A, keypoint_blue_B, intrinsics, X_B_CA, X_B_CB)

In [ ]:
red = (1, 0, 0)
green = (0, 1, 0)
blue = (0, 0, 1)

red_mesh = open3d_point(keypoint_red_3d, red)
green_mesh = open3d_point(keypoint_green_3d, green)
blue_mesh = open3d_point(keypoint_blue_3d, blue)

camera_A_lines = open3d_camera(X_B_CA, camera.intrinsics_matrix(), camera.resolution, cyan)
camera_B_lines = open3d_camera(X_B_CB, camera.intrinsics_matrix(), camera.resolution, magenta)

ray_red_A = open3d_ray_from_pixel(keypoint_red_A, X_B_CA, camera.intrinsics_matrix(), red)
ray_red_B = open3d_ray_from_pixel(keypoint_red_B, X_B_CB, camera.intrinsics_matrix(), red)
ray_green_A = open3d_ray_from_pixel(keypoint_green_A, X_B_CA, camera.intrinsics_matrix(), green)
ray_green_B = open3d_ray_from_pixel(keypoint_green_B, X_B_CB, camera.intrinsics_matrix(), green)
ray_blue_A = open3d_ray_from_pixel(keypoint_blue_A, X_B_CA, camera.intrinsics_matrix(), blue)
ray_blue_B = open3d_ray_from_pixel(keypoint_blue_B, X_B_CB, camera.intrinsics_matrix(), blue)
ray_meshes = [ray_red_A, ray_red_B, ray_green_A, ray_green_B, ray_blue_A, ray_blue_B]

pcd = open3d_pointcloud_from_image_and_depth_map(
    image_B, depth_map_B, camera.intrinsics_matrix(), X_B_CB, camera.resolution
)

visualize_open3D([pcd, camera_A_lines, camera_B_lines, red_mesh, green_mesh, blue_mesh] + ray_meshes)

<div class="alert alert-block alert-info"> <b>Task 19:</b> Use the 3D keypoints to calculate the object frame. Ensure that both the position and orientation of the object are expressed in terms of the keypoints! If we move or rotate the object, the frame should also move/rotate! Make sure your axes are orthonormal and right-handed! Note that we added an additional normalization step to eliminate numerical errors. </div>

In [ ]:
# TODO @students: create the bin frame

P_B_Bin = None # the origin of your bin frame
R_B_BinX = None   # X axis of your bin frame
R_B_BinY = None   # Y axis of your bin frame
R_B_BinZ = None   # Z axis of your bin frame

# additional normalization step to eliminate numerical errors.
R_B_BinZ = R_B_BinZ / np.linalg.norm(R_B_BinZ)
R_B_BinX = np.cross(R_B_BinY, R_B_BinZ)

X_B_BIN = np.identity(4)
X_B_BIN[:3, 0] = R_B_BinX
X_B_BIN[:3, 1] = R_B_BinY
X_B_BIN[:3, 2] = R_B_BinZ
X_B_BIN[:3, 3] = P_B_Bin

bin_pose_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
bin_pose_mesh.transform(X_B_BIN)

base_pose_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)

visualize_open3D([pcd, bin_pose_mesh, base_pose_mesh, red_mesh, green_mesh, blue_mesh])

<div class="alert alert-block alert-info"> <b>Task 20:</b> Make sure your rotation is orthonormal (i.e. you have defined a proper frame in SE3) by running the cell below. </div>

In [ ]:
R_B_BIN = X_B_BIN[:3, :3]

assert np.isclose(np.linalg.inv(R_B_BIN) @ R_B_BIN, np.eye(3)).all() and np.linalg.det(R_B_BIN) == 1, "R_B_BIN is not a valid rotation matrix"

## 5. Grasp frame

<div class="alert alert-block alert-info"> <b>Task 21:</b> Define a grasp frame relative to the object frame. By specifying the graps in the object frame, we make it independent of the actual pose of the object!</div>

In [ ]:
# TODO @students: create the grasp pose relative to the bin frame

X_BIN_GRASP = None # grasp pose expressed in the bin frame


X_B_GRASP = X_B_BIN @ X_BIN_GRASP

grasp_pose_mesh = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)
grasp_pose_mesh.transform(X_B_GRASP)

visualize_open3D([pcd, bin_pose_mesh, base_pose_mesh, grasp_pose_mesh, red_mesh, green_mesh, blue_mesh])

## 6. Grasp execution

<div class="alert alert-block alert-info"> <b>Task 22:</b> Execute the grasp by running the cell below.</div>

<!-- **TODO**: pregrasp pose currently only works when the bin is flat (no on its side). -->

In [ ]:
X_B_PREGRASP = X_B_GRASP.copy()
X_B_PREGRASP[:3, 3] += np.array([0.0, 0.0, 0.07])

robot.move_to_joint_configuration(joints_view_B).wait()

gripper_awaitable = gripper.open()
robot.move_to_tcp_pose(X_B_PREGRASP).wait()
gripper_awaitable.wait()
robot.move_linear_to_tcp_pose(X_B_GRASP).wait()
gripper.close().wait()
robot.move_linear_to_tcp_pose(X_B_PREGRASP).wait()

IF everything went according to plan, you should now have lifted the bin. Congrats! Now te ensure that all your frames are properly defined, change the pose (just make sure all keypoints are still visible in both camera viewpoints) of the bin and rerun the annotation, triangulation and frame construction cells. The object should again be lifted! This illustrates how you can create robots that can generalize across object poses, which is a first step into making them useful in unstructured environments!

<div class="alert alert-block alert-info"> <b>Task 23:</b> Change the pose of the bin and rerun the annotation, triangulation, and frame construction cells. Verify that the object is lifted again. </div>